**Implementing Federated Transfer Learning (FTL) **

1. Pre-training on Client 1 (Dataset A)
2. Fine-tuning on Client 2 (Dataset B)
3. Secret sharing simulation
4. Accuracy comparison with isolated training

**The notebook demonstrates**

1. Dataset loading
2. Splitting into two clients (A & B)
3. Isolated model training
4. Pre-training on Client 1
5. Secret sharing of model weights
6. Secure reconstruction at Client
7. Fine-tuning and evaluation
8. Accuracy comparison table


# Federated Transfer Learning (FTL) Simulation
### Pretraining + Fine-tuning with Privacy Preserving Mechanism

This notebook demonstrates:

1. Client 1 trains a model on Dataset A
2. Model is securely shared (without raw data)
3. Client 2 fine‑tunes the model on Dataset B
4. Compare performance with isolated training
5. Demonstrate a simple Secret Sharing / Pseudo Homomorphic aggregation concept


## 1 Import Libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

## 2 Load Dataset
We split the dataset into two logical datasets representing two clients.

In [ ]:
data = load_breast_cancer()

X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Simulate two clients
split = int(0.5 * len(X_train))

XA = X_train[:split]
yA = y_train[:split]

XB = X_train[split:]
yB = y_train[split:]

print('Client 1 dataset size:', XA.shape)
print('Client 2 dataset size:', XB.shape)

Client 1 dataset size: (227, 30)
Client 2 dataset size: (228, 30)


## 3 Isolated Training (Baseline)
Client 2 trains only on its own dataset.

In [ ]:
model_iso = LogisticRegression(max_iter=500)

model_iso.fit(XB, yB)

pred_iso = model_iso.predict(X_test)

acc_iso = accuracy_score(y_test, pred_iso)

print('Accuracy (Isolated Training):', acc_iso)

Accuracy (Isolated Training): 0.956140350877193


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 4 Client 1 Pre‑Training

In [ ]:
model_A = LogisticRegression(max_iter=500)

model_A.fit(XA, yA)

print('Client 1 pretraining completed')

Client 1 pretraining completed


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 5 Secret Sharing Simulation

Weights are split into random shares before sending.

In [ ]:
weights = model_A.coef_

# generate random share
share1 = np.random.rand(*weights.shape)

# second share
share2 = weights - share1

print('Secret shares created')

Secret shares created


## 6 Secure Reconstruction (Client 2)
Client 2 reconstructs the model weights.

In [ ]:
reconstructed_weights = share1 + share2

model_ftl = LogisticRegression(max_iter=500)

model_ftl.fit(XB, yB)

# replace weights with transferred weights for simulation
model_ftl.coef_ = reconstructed_weights

pred_ftl = model_ftl.predict(X_test)

acc_ftl = accuracy_score(y_test, pred_ftl)

print('Accuracy (FTL):', acc_ftl)

Accuracy (FTL): 0.9473684210526315


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 7 Result Comparison

In [ ]:
results = pd.DataFrame({
    'Method':['Isolated Training','Federated Transfer Learning'],
    'Accuracy':[acc_iso, acc_ftl]
})

results

,Method,Accuracy
0,Isolated Training,0.956140
1,Federated Transfer Learning,0.947368


## Conclusion

Federated Transfer Learning allows knowledge transfer between clients
without sharing raw data. This improves model performance while
preserving privacy.